In [1]:
from pathlib import Path
import sys

# Find project root whether the notebook starts from /notebooks or project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print("Backend path:", BACKEND_DIR)

Backend path: c:\Users\User\Desktop\Genetic-Algorithm Dashboard\backend


In [ ]:


import copy
import pickle
import os

from datetime import datetime

from genetic_algorithm.utils.Functions import (
    plot_schedule,
    list_subjects,
    lst_rooms,
    lst_sections,
    list_faculty,
    create_schedule_with_minimum_load,
)

from genetic_algorithm.operators.BestChromosome import (
    best_chromosome_to_dataframe,
    compare_faculty_preferences,
)

from genetic_algorithm.operators.mutation import (
    mutate_by_swapping_faculty_subjects,
)

from genetic_algorithm.operators.CreatePopulation import (
    generate_population,
)

POPULATION_SIZE = 100

chromosomes = generate_population(
    population_size=POPULATION_SIZE,
    list_faculty=list_faculty,
    list_subjects=list_subjects,
    lst_rooms=lst_rooms,
    max_restarts=2000
)

ModuleNotFoundError: No module named 'Functions'

In [3]:


from genetic_algorithm.utils.Functions import df_faculty_pref

from genetic_algorithm.operators.FitnessFunction import (
    faculty_preference_fitness
)

from genetic_algorithm.operators.ElitisimSelection import (
    elitism_selection
)

from genetic_algorithm.operators.Swap import (
    create_child_by_faculty_swap
)

from genetic_algorithm.operators.CreatePopulation import (
    create_new_population,
    get_top_population,
    sort_population_by_fitness
)

from genetic_algorithm.operators.BestChromosome import (
    best_chromosome_to_dataframe,
    compare_faculty_preferences
)

from genetic_algorithm.analysis.FacultySelectedChromosomeDashboard import (
    visualize_selected_chromosome_dashboard
)

from genetic_algorithm.analysis.FacultyFitnessVisualization import (
    visualize_faculty_fitness,
    display_faculty_fitness_summary
)


population = chromosomes

for iii in range(0,10000):
    fitness_scores = [
        faculty_preference_fitness(
            chromosome,
            df_faculty_pref
        )
        for chromosome in population
    ]
    
    (elites,
        elite_fitness,
        non_elites,
        non_elite_fitness
    ) = elitism_selection(
        population=population,
        fitness_scores=fitness_scores,
        elite_percentage=0.10
    )

    fitness_scores = [
        faculty_preference_fitness(
            chromosome,
            df_faculty_pref
        )
        for chromosome in non_elites
    ]


    (
        top_non_elites,
        elite_fitness1,
        non_elites2,
        non_elite_fitness
    ) = elitism_selection(
        population=non_elites,
        fitness_scores=fitness_scores,
        elite_percentage=0.5
    )


    new_population = []
    num = POPULATION_SIZE * .9
    while len(new_population) < int(num):
        result = create_child_by_faculty_swap(
            population=top_non_elites,
            df_faculty_pref=df_faculty_pref,
            max_attempts=100
        )
        if result != 'None':
            new_population.append(result)

    for i in top_non_elites:
        if i not in new_population:
            new_population.append(i)

    tmp = []
    for i in new_population:
        try:
            z = len(i)
            tmp.append(i)
        except:
            pass
    new_population = tmp

    for i in range(0,1):

        result = mutate_by_swapping_faculty_subjects(
            population=new_population,
            df_faculty_pref=df_faculty_pref,
            max_attempts=10
        )
        if result != "None":
            new_population.append(result)


    for i in elites:
        if i not in new_population:
            new_population.append(i)
    tmp = []
    for i in new_population:
        try:
            z = len(i)
            tmp.append(i)
        except:
            pass
    new_population = tmp

    new_chromosomes = generate_population(
        population_size=50,
        list_faculty=list_faculty,
        list_subjects=list_subjects,
        lst_rooms=lst_rooms,
        max_restarts=300
    )
    for i in new_chromosomes :
        if i not in new_population:
            new_population.append(i)

    tmp = []
    for i in new_population:
        try:
            z = len(i)
            tmp.append(i)
        except:
            pass
    new_population = tmp
    
    new_population = sort_population_by_fitness(
        new_population,
        df_faculty_pref
    )
    best_chromosome = new_population[0]

    df = best_chromosome_to_dataframe(best_chromosome)
    df_faculty_pref.to_csv('Faculty_preferences.csv')
    df.to_csv('Best_chromosome.csv')
    result = compare_faculty_preferences(
        'Faculty_preferences.csv',
        'Best_chromosome.csv'
    )
    df1 = result[0]
    df2 = result[1]
    df3 = result[2]
    average_overall = str(sum(list(result[1]['Overall_Match']))/ len(list(result[1]['Overall_Match'])))

    # 2. Save the list to a file
    fitness_score = faculty_preference_fitness(  best_chromosome, df_faculty_pref )
    dt = str(datetime.now()).replace('-','').replace(':','').replace(' ','').replace('.','')
    folder_name = str(fitness_score) +'_'+ dt
    os.mkdir(folder_name)
    with open(folder_name + '/best_chromosome.pkl', 'wb') as file:
        pickle.dump(best_chromosome, file)
    # with open(folder_name + '/new_population.pkl', 'wb') as file:
    #     pickle.dump(new_population, file)
    df1.to_csv(folder_name +'/df1.csv')
    df2.to_csv(folder_name +'/df2.csv')
    df3.to_csv(folder_name +'/df3.csv')

    tmp = []
    for i in new_population[:POPULATION_SIZE+50]:
        try:
            z = len(i)
            tmp.append(i)
        except:
            pass
    population = tmp
    
    if iii % 3 == 0:

        faculty_results, daily_results, summary, fig = (
            visualize_selected_chromosome_dashboard(
                best_chromosome,
                df_faculty_pref,
                chromosome_name="Best Chromosome"
            )
)
        pass

ModuleNotFoundError: No module named 'pandas'

In [ ]:
with open('2735_20260916225151171122/best_chromosome.pkl', 'rb') as file:
    loaded_list = pickle.load(file)

In [ ]:
best_chromosome = loaded_list

In [ ]:
# from GetUpdatedList import  get_updated_lists
# updated_list_rooms, updated_list_faculty = get_updated_lists(population [0])